# getitem-back-add-at — faded example 2: Fill the gradient buffer allocation

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `getitem-back-add-at`. Running the beacon reports progress on the `Backprop: getitem_back via add-at` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: getitem_back via add-at` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`getitem-back-add-at`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "getitem-back-add-at"
DD_SUBTOPIC = "Backprop: getitem_back via add-at"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

The gradient of an index gather has the shape of the SOURCE `x`, not of the gathered output. Allocating `zeros_like(x)` before the scatter-add is what gives the gradient the right `(N, D)` shape for the embedding backward.

## Faded exercise 2

Implement `getitem_back_rows(grad_out, x, idx)` for a row gather `out = x[idx]` with `x: (N, D)`. Allocate the gradient buffer with the shape of `x`, then scatter-add the rows of `grad_out`. Complete the blanked allocation.

**Fill in:** the zeros buffer shaped like the source x (N, D) to scatter row gradients into

In [ ]:
import torch as t

t.manual_seed(4)
x = t.zeros(4, 3)
idx = t.tensor([0, 2, 2])
grad_out = t.tensor([[1.0, 0.0, 0.0], [0.0, 1.0, 0.0], [0.0, 0.0, 1.0]])

def getitem_back_rows(grad_out, x, idx):
    grad_in = t.zeros_like(x)
    grad_in.index_add_(0, idx, grad_out)
    return grad_in

print(getitem_back_rows(grad_out, x, idx).tolist())


def _test():
    gi = getitem_back_rows(grad_out, x, idx)
    # gradient must have the SOURCE shape, not the gathered (K, D) shape
    assert gi.shape == (4, 3), gi.shape
    # independent ground truth via autograd
    xr = t.zeros(4, 3, requires_grad=True)
    out = xr[idx]
    (out * grad_out).sum().backward()
    assert t.allclose(gi, xr.grad), (gi, xr.grad)
    # row 2 (repeated) accumulates both contributions
    assert gi[2].tolist() == [0.0, 1.0, 1.0]


try:
    _test()
    _dd_passed.add('faded2')
    print('[Delta Drills] faded2 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t

t.manual_seed(4)
x = t.zeros(4, 3)
idx = t.tensor([0, 2, 2])
grad_out = t.tensor([[1.0, 0.0, 0.0], [0.0, 1.0, 0.0], [0.0, 0.0, 1.0]])

def getitem_back_rows(grad_out, x, idx):
    grad_in = t.zeros_like(x)
    grad_in.index_add_(0, idx, grad_out)
    return grad_in

print(getitem_back_rows(grad_out, x, idx).tolist())
```
</details>